In [ ]:
%pip install h3 ipyleaflet

In [ ]:
import random
import h3
import math
import json
from ipyleaflet import Map, Polygon, CircleMarker, Popup
from IPython.display import display
from ipywidgets import Layout, HTML

In [ ]:
RESOLUTION = 8
SEARCH_LAT, SEARCH_LNG = 23.8103, 90.4125
USER_BASE_LAT_MIN = 21.5
USER_BASE_LAT_MAX = 26.3
USER_BASE_LNG_MIN = 88.5
USER_BASE_LNG_MAX = 92.2

In [ ]:
def generate_user(i):
    """Generates a random user within Bangladesh bounds."""
    lat = random.uniform(USER_BASE_LAT_MIN, USER_BASE_LAT_MAX)
    lng = random.uniform(USER_BASE_LNG_MIN, USER_BASE_LNG_MAX)

    h3_hex = h3.latlng_to_cell(lat, lng, RESOLUTION)

    return {
        "id": i,
        "name": f"User_{i}",
        "lat": lat,
        "lng": lng,
        "h3_index": h3_hex
    }

In [ ]:
def populate_data(n):

    users = (generate_user(i) for i in range(n))
    data = {}
    for u in users:
        h3_key = u['h3_index']
        if h3_key not in data:
            data[h3_key] = []
        data[h3_key].append(u)

    return data

In [ ]:
def find_nearby_users(search_lat, search_lng, radius_km, data):

    center_hex = h3.latlng_to_cell(search_lat, search_lng, RESOLUTION)
    edge_len = h3.average_hexagon_edge_length(RESOLUTION, unit='km')

    k_required = math.ceil(radius_km / (edge_len * 1.5))
    search_hexes = h3.grid_disk(center_hex, k_required)

    nearby_index = {}

    for h_hex in search_hexes:

        cell_users = data.get(h_hex, [])

        if cell_users:

            filtered_users = [
                u for u in cell_users
                if h3.great_circle_distance((search_lat, search_lng), (u['lat'], u['lng'])) <= radius_km
            ]

            if filtered_users:
                nearby_index[h_hex] = filtered_users

    return nearby_index

In [ ]:
def plot_indexed_users(results_dict, m, match=False):
    """
    Plots users from a dictionary where keys are H3 Hex Strings.
    results_dict: { "8828308281fffff": [user_dict, ...] }
    """

    main_color = "green" if match else "red"

    for h3_hex, user_list in results_dict.items():

        to_remove = next((l for l in m.layers if getattr(
            l, 'name', None) == h3_hex), None)

        if to_remove:
            continue

        boundary_coords = [list(pt) for pt in h3.cell_to_boundary(h3_hex)]

        hex_poly = Polygon(
            locations=boundary_coords,
            color=main_color,
            fill_color=main_color,
            fill_opacity=0.1,
            weight=1,
            name=f"Hex {h3_hex}"
        )
        m.add_layer(hex_poly)

        for user in user_list:
            user_info = HTML(value=f"""
                <div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; padding: 5px;">
                    <strong style="color: {main_color};">User ID:</strong> {user['id']}<br>
                    <strong>H3:</strong> <code>{h3_hex}</code><br>
                    <hr style="border: 0; border-top: 1px solid #ddd; margin: 8px 0;">
                    <span style="font-size: 0.85em; color: #666;">
                        Lat: {user['lat']:.5f}<br>Lng: {user['lng']:.5f}
                    </span>
                </div>
            """)

            user_popup = Popup(
                location=(user['lat'], user['lng']),
                child=user_info,
                close_button=True,
                auto_close=True,
                auto_pan=False
            )

            dot = CircleMarker(
                location=(user['lat'], user['lng']),
                radius=4,
                color=main_color,
                fill_color=main_color,
                fill_opacity=0.7,
                popup=user_popup
            )
            m.add_layer(dot)

In [ ]:
map_layout = Layout(width='100vw', height='700px')
display_map = Map(center=(SEARCH_LAT, SEARCH_LNG), zoom=6,
                  scroll_wheel_zoom=True, layout=map_layout)
display(display_map)

In [ ]:
search_data = populate_data(1_000)
plot_indexed_users(search_data, display_map, False)

radius = 6
results = find_nearby_users(SEARCH_LAT, SEARCH_LNG, radius, search_data)

count = sum(len(users) for users in results.values())

plot_indexed_users(results, display_map, True)

print(f"Found {count} users within {radius}km of Dhaka center.")